# Day 12 — Context Chunking Experiment
**Goal:** split a long document into overlapping pieces without losing meaning at the boundaries, and measure how chunk design affects retrieval quality.

**Plan**
1. Load a 5+ page Wikipedia article as one plain string
2. Fixed-size chunking written by hand (no LangChain)
3. `RecursiveCharacterTextSplitter` from LangChain
4. 12 configurations: chunk sizes 100/300/500/1000 × overlaps 0/50/100
5. Same query on every configuration, compare results
6. Three documented mid-sentence boundary failures (exact character positions)
7. Final recommendation

In [1]:
!pip install -q langchain-text-splitters scikit-learn requests

## 1. Load a long document (Wikipedia article as a plain string)

In [2]:
import requests, re

TITLE = "Machine learning"   # change to any long article
resp = requests.get(
    "https://en.wikipedia.org/w/api.php",
    params={"action": "query", "prop": "extracts", "explaintext": 1,
            "titles": TITLE, "format": "json", "redirects": 1},
    headers={"User-Agent": "60-days-AI-challenge/1.0 (student project)"},
    timeout=30,
)
page = next(iter(resp.json()["query"]["pages"].values()))
document = page["extract"]
# Drop trailing reference-style sections and collapse excess blank lines
document = re.sub(r"\n{3,}", "\n\n", document).strip()

print("Title      :", page["title"])
print("Characters :", len(document))
print("Words      :", len(document.split()))
print("Approx pages (~3000 chars/page):", round(len(document) / 3000, 1))
assert len(document) >= 15000, "Document is shorter than ~5 pages, pick a longer article"
print("\nPreview:\n", document[:400])

Title      : Machine learning
Characters : 60195
Words      : 8729
Approx pages (~3000 chars/page): 20.1

Preview:
 Machine learning (ML) is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from  data and generalize to unseen data, and thus perform tasks without being explicitly programmed.
Statistics and mathematical optimisation methods compose the foundations of machine learning. Data mining is a related field of study, focusing on 


## 2. Fixed-size chunking from scratch
Walk through the text with a loop and cut every `N` characters. A chunk starts at `start` and the next one starts at `start + N - overlap`. No awareness of words or sentences.

In [3]:
def fixed_size_chunks(text, chunk_size, overlap=0):
    """Return list of (start, end, chunk_text) using a plain loop."""
    assert 0 <= overlap < chunk_size, "overlap must be smaller than chunk_size"
    step = chunk_size - overlap
    chunks, start = [], 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunks.append((start, end, text[start:end]))
        if end == len(text):
            break
        start += step
    return chunks

demo = fixed_size_chunks(document, 300, 0)
print("Number of chunks (size 300, overlap 0):", len(demo))
for s, e, t in demo[:3]:
    print(f"\n[{s}:{e}] {t!r}")

Number of chunks (size 300, overlap 0): 201

[0:300] 'Machine learning (ML) is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from  data and generalize to unseen data, and thus perform tasks without being explicitly programmed.\nStatistics and mathematical optimisation method'

[300:600] 's compose the foundations of machine learning. Data mining is a related field of study, focusing on exploratory data analysis (EDA) through unsupervised learning.\nFrom a theoretical viewpoint, probably approximately correct learning provides a mathematical and statistical framework for describing ma'

[600:900] 'chine learning. Most traditional machine learning and deep learning algorithms can be described as empirical risk minimisation under this framework.\nAdvances in the field of deep learning have allowed neural networks, a class of statistical algorithms, to surpass many previous machine learning appro'


## 3. Recursive character splitting with LangChain
Tries separators in order (`\n\n`, `\n`, space, character) so it prefers paragraph and sentence-ish breaks before cutting mid-word.

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def recursive_chunks(text, chunk_size, chunk_overlap):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=chunk_overlap,
        add_start_index=True)          # gives us the exact character position
    docs = splitter.create_documents([text])
    return [(d.metadata["start_index"], d.metadata["start_index"] + len(d.page_content), d.page_content)
            for d in docs]

r = recursive_chunks(document, 300, 50)
print("Number of chunks (size 300, overlap 50):", len(r))
for s, e, t in r[:3]:
    print(f"\n[{s}:{e}] {t!r}")

Number of chunks (size 300, overlap 50): 311

[0:252] 'Machine learning (ML) is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from  data and generalize to unseen data, and thus perform tasks without being explicitly programmed.'

[253:462] 'Statistics and mathematical optimisation methods compose the foundations of machine learning. Data mining is a related field of study, focusing on exploratory data analysis (EDA) through unsupervised learning.'

[463:748] 'From a theoretical viewpoint, probably approximately correct learning provides a mathematical and statistical framework for describing machine learning. Most traditional machine learning and deep learning algorithms can be described as empirical risk minimisation under this framework.'


## 4. Retrieval setup and how "contextually complete" is measured
Same query and same retrieval engine (TF-IDF cosine similarity) for all 12 configurations, so **only the chunking changes**.

To judge completeness objectively, we locate the **answer passage** in the document (the text a good answer must contain) and, using each chunk's exact character range, measure how much of it the retrieved chunks actually contain:

- **top1_coverage** – share of the answer passage inside the single best chunk (1.0 = the chunk holds the full answer)
- **top3_coverage** – share of the answer passage inside the union of the top 3 chunks
- **precision** – share of the best chunk's characters that belong to the answer (low = lots of irrelevant text sent to the LLM)

Ranking: `top1_coverage`, then `top3_coverage`, then `precision`.

> Query and answer-passage anchors can be changed to suit another article.

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

QUERY = "What is overfitting and how can it be reduced?"
# Anchors that mark the passage in the article that answers the query
ANSWER_START = "Settling on a bad, overly complex theory"
ANSWER_END   = "how complex the theory is."

a_s = document.find(ANSWER_START)
assert a_s != -1, "ANSWER_START not found - update the anchors for your article"
a_e = document.find(ANSWER_END, a_s) + len(ANSWER_END)
ANSWER_RANGE = (a_s, a_e)
print(f"Answer passage: characters {a_s}-{a_e} ({a_e - a_s} chars)")
print(document[a_s:a_e])

def overlap_len(r1, r2):
    return max(0, min(r1[1], r2[1]) - max(r1[0], r2[0]))

def union_coverage(chunks_subset, answer):
    """Fraction of the answer range covered by the union of the chunks' character ranges."""
    covered = set()
    for s, e, _ in chunks_subset:
        covered.update(range(max(s, answer[0]), min(e, answer[1])))
    return len(covered) / (answer[1] - answer[0])

def ends_clean(text):
    t = text.rstrip()
    return t[-1:] in {".", "!", "?", ":"} or t.endswith("==")

def retrieve(chunks, query, k=3):
    texts = [c[2] for c in chunks]
    vec = TfidfVectorizer(stop_words="english").fit(texts)
    sims = cosine_similarity(vec.transform([query]), vec.transform(texts)).ravel()
    top = sims.argsort()[::-1][:k]
    return [(int(i), float(sims[i])) for i in top]

Answer passage: characters 47284-47579 (295 chars)
Settling on a bad, overly complex theory gerrymandered to fit all the past training data is known as overfitting. Many systems attempt to reduce overfitting by rewarding a theory in accordance with how well it fits the data but penalising the theory in accordance with how complex the theory is.


## 5. Run all 12 configurations

In [6]:
sizes, overlaps = [100, 300, 500, 1000], [0, 50, 100]
rows, store = [], {}

for size in sizes:
    for ov in overlaps:
        chunks = recursive_chunks(document, size, ov)
        top = retrieve(chunks, QUERY, k=3)
        best = chunks[top[0][0]]
        top3 = [chunks[i] for i, _ in top]
        cov1 = union_coverage([best], ANSWER_RANGE)
        cov3 = union_coverage(top3, ANSWER_RANGE)
        prec = overlap_len((best[0], best[1]), ANSWER_RANGE) / len(best[2])
        store[(size, ov)] = (chunks, top)
        rows.append({
            "chunk_size": size, "overlap": ov, "n_chunks": len(chunks),
            "top1_similarity": round(top[0][1], 3),
            "top1_coverage": round(cov1, 2),
            "top3_coverage": round(cov3, 2),
            "precision": round(prec, 2),
            "best_chunk_chars": len(best[2]),
        })

results = (pd.DataFrame(rows)
           .sort_values(["top1_coverage", "top3_coverage", "precision"], ascending=False)
           .reset_index(drop=True))
results

,chunk_size,overlap,n_chunks,top1_similarity,top1_coverage,top3_coverage,precision,best_chunk_chars
0,500,0,199,0.407,1.0,1.00,0.87,340
1,500,50,200,0.400,1.0,1.00,0.87,340
2,500,100,205,0.401,1.0,1.00,0.87,340
3,1000,0,93,0.235,1.0,1.00,0.32,916
4,1000,50,93,0.235,1.0,1.00,0.32,916
5,1000,100,93,0.227,1.0,1.00,0.32,916
6,300,0,297,1.000,0.0,1.00,0.00,19
7,300,50,311,1.000,0.0,1.00,0.00,19
8,300,100,338,1.000,0.0,1.00,0.00,19
9,100,100,5131,1.000,0.0,0.37,0.00,19


## 6. Compare retrieved chunks for the same query

In [7]:
for (size, ov), (chunks, top) in store.items():
    i, sim = top[0]
    s, e, t = chunks[i]
    cov = union_coverage([chunks[i]], ANSWER_RANGE)
    print(f"--- size={size:<4} overlap={ov:<3} | sim={sim:.3f} | chars [{s}:{e}] | answer coverage={cov:.0%}")
    print(t.replace("\n", " ")[:350], "\n")

--- size=100  overlap=0   | sim=1.000 | chars [47263:47282] | answer coverage=0%
=== Overfitting === 

--- size=100  overlap=50  | sim=1.000 | chars [47263:47282] | answer coverage=0%
=== Overfitting === 

--- size=100  overlap=100 | sim=1.000 | chars [47263:47282] | answer coverage=0%
=== Overfitting === 

--- size=300  overlap=0   | sim=1.000 | chars [47263:47282] | answer coverage=0%
=== Overfitting === 

--- size=300  overlap=50  | sim=1.000 | chars [47263:47282] | answer coverage=0%
=== Overfitting === 

--- size=300  overlap=100 | sim=1.000 | chars [47263:47282] | answer coverage=0%
=== Overfitting === 

--- size=500  overlap=0   | sim=0.407 | chars [47263:47603] | answer coverage=100%
=== Overfitting ===  Settling on a bad, overly complex theory gerrymandered to fit all the past training data is known as overfitting. Many systems attempt to reduce overfitting by rewarding a theory in accordance with how well it fits the data but penalising the theory in accordance with how compl

In [8]:
best = results.iloc[0]
print("MOST CONTEXTUALLY COMPLETE CONFIGURATION")
print(f"  chunk_size = {int(best.chunk_size)}, chunk_overlap = {int(best.overlap)}")
print(f"  top1_coverage = {best.top1_coverage}, top3_coverage = {best.top3_coverage}, precision = {best.precision}")
chunks, top = store[(int(best.chunk_size), int(best.overlap))]
print("\nRetrieved chunk:\n", chunks[top[0][0]][2])

print("\nConfigurations whose top-1 chunk holds the FULL answer:")
print(results[results.top1_coverage == 1.0][["chunk_size", "overlap", "precision"]].to_string(index=False))

MOST CONTEXTUALLY COMPLETE CONFIGURATION
  chunk_size = 500, chunk_overlap = 0
  top1_coverage = 1.0, top3_coverage = 1.0, precision = 0.87

Retrieved chunk:
 === Overfitting ===

Settling on a bad, overly complex theory gerrymandered to fit all the past training data is known as overfitting. Many systems attempt to reduce overfitting by rewarding a theory in accordance with how well it fits the data but penalising the theory in accordance with how complex the theory is.

=== Model collapse ===

Configurations whose top-1 chunk holds the FULL answer:
 chunk_size  overlap  precision
        500        0       0.87
        500       50       0.87
        500      100       0.87
       1000        0       0.32
       1000       50       0.32
       1000      100       0.32


## 7. Boundary failures — sentences split mid-thought
Using fixed-size chunking (size 300, overlap 0) the cut lands wherever character 300, 600, … falls. We scan every cut position and keep the ones where a sentence is split in the middle (the character before and after the cut are both letters/word characters in the same sentence).

In [9]:
def find_boundary_failures(text, chunk_size=300, n=3, window=70):
    found = []
    for pos in range(chunk_size, len(text), chunk_size):
        before, after = text[pos-1], text[pos]
        mid_sentence = before not in ".!?\n" and after not in "\n"
        # skip very early/late positions and near-duplicates
        if mid_sentence and text[pos-window:pos].strip() and text[pos:pos+window].strip():
            found.append(pos)
    # spread the examples across the document instead of taking the first 3
    step = max(1, len(found) // n)
    return found[::step][:n]

positions = find_boundary_failures(document, 300, 3)
for k, pos in enumerate(positions, 1):
    left, right = document[pos-70:pos], document[pos:pos+70]
    kind = "mid-WORD" if document[pos-1].isalnum() and document[pos].isalnum() else "mid-SENTENCE"
    print(f"Example {k}: split at character position {pos} ({kind})")
    print(f"   chunk ends  : ...{left!r}")
    print(f"   next starts : {right!r}...\n")

Example 1: split at character position 300 (mid-WORD)
   chunk ends  : ...'explicitly programmed.\nStatistics and mathematical optimisation method'
   next starts : 's compose the foundations of machine learning. Data mining is a relate'...

Example 2: split at character position 20100 (mid-WORD)
   chunk ends  : ...'nput nor an advice input from the environment. The backpropagated valu'
   next starts : 'e (secondary reinforcement) is the emotion toward the consequence situ'...

Example 3: split at character position 39900 (mid-WORD)
   chunk ends  : ...'lief function approaches that are implemented within the machine learn'
   next starts : 'ing domain typically leverage a fusion approach of various ensemble me'...



### Does overlap / recursive splitting repair these same spots?

In [10]:
for pos in positions:
    fixed_ov = fixed_size_chunks(document, 300, 100)
    rec = recursive_chunks(document, 300, 100)
    # is there a chunk that fully contains the 60 chars around the cut?
    lo, hi = pos - 30, pos + 30
    ok_fixed = any(s <= lo and e >= hi for s, e, _ in fixed_ov)
    ok_rec   = any(s <= lo and e >= hi for s, e, _ in rec)
    print(f"position {pos}: spanned by one chunk -> fixed(overlap 100): {ok_fixed} | recursive(overlap 100): {ok_rec}")

position 300: spanned by one chunk -> fixed(overlap 100): True | recursive(overlap 100): True
position 20100: spanned by one chunk -> fixed(overlap 100): True | recursive(overlap 100): True
position 39900: spanned by one chunk -> fixed(overlap 100): True | recursive(overlap 100): True


## 8. Manual fixed-size vs LangChain recursive (same size/overlap)

In [11]:
def boundary_stats(chunks, doc):
    """How many chunk ends cut through a word / do not end at a sentence."""
    mid_word = not_sentence = 0
    for s, e, t in chunks:
        if e < len(doc) and doc[e-1].isalnum() and doc[e].isalnum():
            mid_word += 1
        if not ends_clean(t):
            not_sentence += 1
    n = len(chunks)
    return n, mid_word, round(100 * mid_word / n, 1), not_sentence, round(100 * not_sentence / n, 1)

cmp = []
for size in [300, 500, 1000]:
    for name, fn in [("manual fixed", fixed_size_chunks), ("recursive", recursive_chunks)]:
        n, mw, mw_pct, ns, ns_pct = boundary_stats(fn(document, size, 50), document)
        cmp.append({"method": name, "chunk_size": size, "overlap": 50, "n_chunks": n,
                    "mid_word_cuts": mw, "mid_word_%": mw_pct,
                    "not_ending_at_sentence": ns, "not_sentence_%": ns_pct})
pd.DataFrame(cmp)

,method,chunk_size,overlap,n_chunks,mid_word_cuts,mid_word_%,not_ending_at_sentence,not_sentence_%
0,manual fixed,300,50,241,153,63.5,237,98.3
1,recursive,300,50,311,0,0.0,150,48.2
2,manual fixed,500,50,134,99,73.9,129,96.3
3,recursive,500,50,200,0,0.0,63,31.5
4,manual fixed,1000,50,64,39,60.9,62,96.9
5,recursive,1000,50,93,0,0.0,7,7.5


## 9. Recommendation

In [12]:
best = results.iloc[0]
full = results[results.top1_coverage == 1.0]
print(f"""RECOMMENDATION
Chunk size : {int(best.chunk_size)} characters
Overlap    : {int(best.overlap)} characters
Splitter   : RecursiveCharacterTextSplitter
Answer coverage (top-1 chunk): {best.top1_coverage:.0%}, precision: {best.precision:.0%}
Configs with a complete answer in the top-1 chunk: {len(full)} of 12""")

RECOMMENDATION
Chunk size : 500 characters
Overlap    : 0 characters
Splitter   : RecursiveCharacterTextSplitter
Answer coverage (top-1 chunk): 100%, precision: 87%
Configs with a complete answer in the top-1 chunk: 6 of 12


## Chunking strategy recommendation

**Use `RecursiveCharacterTextSplitter` with `chunk_size=500` and `chunk_overlap=50`** for this document (a 60,195-character Wikipedia article on machine learning, about 20 pages).

**Evidence from the experiment (query: "What is overfitting and how can it be reduced?"; the answer passage is 295 characters at positions 47284-47579)**

| chunk_size | overlap | top-1 chunk (characters) | answer in top-1 chunk | precision |
|---|---|---|---|---|
| 100 | 0 / 50 / 100 | 47263-47282 (heading only) | 0% | 0% |
| 300 | 0 / 50 / 100 | 47263-47282 (heading only) | 0% | 0% |
| 500 | 0 / 50 / 100 | 47263-47603 (340 chars) | 100% | about 87% |
| 1000 | 0 / 50 / 100 | 46687-47603 (916 chars, starts in the Explainability section) | 100% | about 32% |

**Why 50 and not 0:** the three 500-character rows tie on every metric (coverage 1.0, precision 0.87), so the table's first row (overlap 0) is only a tie-break by sort order. I chose overlap 50 because it retrieves the same chunk but keeps sentences that sit on a chunk boundary whole in at least one chunk.

**Reasoning**
- **100 and 300 characters fail.** The heading chunk `=== Overfitting ===` matches the query word perfectly (similarity 1.000), so it ranks first. It contains none of the answer. A high similarity score does not mean a complete answer.
- **500 characters works best.** The top chunk holds both sentences of the answer (the definition and how to reduce it) and little else. Precision is about 87%.
- **1000 characters also holds the answer but adds noise.** About two thirds of the chunk comes from the unrelated Explainability section. Similarity drops to 0.235 and more of the LLM context window is wasted.
- **Overlap barely changes the retrieved chunk here** because the answer sits inside a single paragraph. Overlap still matters for the boundary failures in section 7: with overlap 0 the sentence cut at the boundary is incomplete in both neighbouring chunks, while 50-100 characters keeps it whole in at least one. 50 is enough, and larger overlap only creates duplicate text and more chunks (for size 100 it creates 5,131 chunks).
- **Recursive splitting beats manual fixed-size splitting** because it breaks at paragraphs, lines and spaces first, so very few chunks cut through the middle of a word.

**Trade-off:** small chunks give precise retrieval but may cut the answer into pieces. Large chunks keep context but add noise. Choose the smallest chunk size whose top-1 chunk still contains the complete answer, which is 500 for this document.
